### Retail Data Analysis

In [2]:
import pandas as pd
import matplotlib.pyplot as plt

In [9]:
customer_data = pd.read_csv('../dataset/customer_profiles.csv')
transaction_data = pd.read_csv('../dataset/sales_transaction.csv')
inventory_data = pd.read_csv('../dataset/product_inventory.csv')

#### Identify products with the highest and lowest sales to inform inventory decisions

In [31]:
transaction_inventory_data = pd.merge(transaction_data,inventory_data, how="inner")
transaction_inventory_data

,TransactionID,CustomerID,ProductID,QuantityPurchased,TransactionDate,Price,ProductName,Category,StockLevel
0,1,103,120,3,01/01/23,30.43,Product_120,Electronics,354
1,2,436,126,1,01/01/23,15.19,Product_126,Electronics,82
2,3,861,55,3,01/01/23,67.76,Product_55,Electronics,135
3,4,271,27,2,01/01/23,65.77,Product_27,Electronics,324
4,5,107,118,1,01/01/23,14.55,Product_118,Beauty & Health,44
...,...,...,...,...,...,...,...,...,...
4977,4998,451,80,1,28/07/23,58.28,Product_80,Electronics,90
4978,4999,904,188,2,28/07/23,36.22,Product_188,Beauty & Health,86
4979,5000,215,159,2,28/07/23,17.42,Product_159,Beauty & Health,232
4980,4999,904,188,2,28/07/23,36.22,Product_188,Beauty & Health,86


In [26]:
transaction_inventory_data.groupby("ProductName")['TransactionID'].count().reset_index(name="count").sort_values(by='count', ascending=False)

,ProductName,count
78,Product_17,39
92,Product_182,38
185,Product_87,35
115,Product_22,35
97,Product_187,34
...,...,...
41,Product_136,15
109,Product_198,15
156,Product_60,14
127,Product_33,12


#### find the sales trends and the m-o-m growth of sales from the dataset.

In [41]:
transaction_inventory_data['TransactionDate'] = pd.to_datetime(transaction_inventory_data['TransactionDate'])

/var/folders/g2/c4ppjm_x53lfmhpz_sqp533r0000gn/T/ipykernel_60726/3821046875.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  transaction_inventory_data['TransactionDate'] = pd.to_datetime(transaction_inventory_data['TransactionDate'])


In [35]:
transaction_inventory_data.groupby("ProductName")['QuantityPurchased'].sum().reset_index(name='count').sort_values(by="count", ascending=False)

,ProductName,count
92,Product_182,102
78,Product_17,100
185,Product_87,92
168,Product_71,88
106,Product_195,87
...,...,...
156,Product_60,35
136,Product_41,35
83,Product_174,33
127,Product_33,31


In [42]:
transaction_inventory_data['OrderMonth'] = transaction_inventory_data['TransactionDate'].dt.to_period("M")

In [44]:
transaction_inventory_data['sale'] = transaction_inventory_data['Price'] * transaction_inventory_data['QuantityPurchased']

In [45]:
transaction_inventory_data.groupby("OrderMonth")['sale'].sum().reset_index(name="total_sales")

,OrderMonth,total_sales
0,2023-01,86110.12
1,2023-02,76081.00
2,2023-03,87712.97
3,2023-04,82792.22
4,2023-05,86775.25
5,2023-06,84097.87
6,2023-07,73477.71
7,2023-08,24427.14
8,2023-09,23744.70
9,2023-10,23003.48


#### Segment customers based on total products purchased and spending to tailor marketing efforts

In [49]:
transaction_inventory_data["prev_purchase_date"] = transaction_inventory_data.groupby("CustomerID")["TransactionDate"].shift(1)
transaction_inventory_data["days_between"] = (transaction_inventory_data["TransactionDate"] - transaction_inventory_data["prev_purchase_date"]).dt.days

In [50]:
transaction_inventory_data

,TransactionID,CustomerID,ProductID,QuantityPurchased,TransactionDate,Price,ProductName,Category,StockLevel,OrderMonth,sale,prev_purchase_date,days_between
0,1,103,120,3,2023-01-01,30.43,Product_120,Electronics,354,2023-01,91.29,NaT,NaN
1,2,436,126,1,2023-01-01,15.19,Product_126,Electronics,82,2023-01,15.19,NaT,NaN
2,3,861,55,3,2023-01-01,67.76,Product_55,Electronics,135,2023-01,203.28,NaT,NaN
3,4,271,27,2,2023-01-01,65.77,Product_27,Electronics,324,2023-01,131.54,NaT,NaN
4,5,107,118,1,2023-01-01,14.55,Product_118,Beauty & Health,44,2023-01,14.55,NaT,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4977,4998,451,80,1,2023-07-28,58.28,Product_80,Electronics,90,2023-07,58.28,2023-07-13,15.0
4978,4999,904,188,2,2023-07-28,36.22,Product_188,Beauty & Health,86,2023-07,72.44,2023-05-31,58.0
4979,5000,215,159,2,2023-07-28,17.42,Product_159,Beauty & Health,232,2023-07,34.84,2023-11-03,-98.0
4980,4999,904,188,2,2023-07-28,36.22,Product_188,Beauty & Health,86,2023-07,72.44,2023-07-28,0.0


#### identify loyal customers based on duration between purchases.

In [51]:
loyalty = (
    transaction_inventory_data.groupby("CustomerID")["days_between"]
      .mean()
      .reset_index()
      .rename(columns={"days_between": "avg_days_between"})
)

In [53]:
loyalty["loyal_customer"] = loyalty["avg_days_between"] <= 30
loyalty

,CustomerID,avg_days_between,loyal_customer
0,1,2.875000,True
1,2,6.800000,True
2,3,-5.333333,True
3,4,20.000000,True
4,5,-52.500000,True
...,...,...,...
984,996,26.333333,True
985,997,21.666667,True
986,998,57.000000,False
987,999,43.250000,False
